In [ ]:
#Workshop: Create Reusable Pipeline
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [1]:
# ---------------------------------------------------------
# 1. สร้าง ข้อมูลจำลอง
# ---------------------------------------------------------
np.random.seed(42)
n_samples = 1000

# สร้าง Features
age = np.random.choice([25, 30, 35, 40, np.nan], size=n_samples)
income = np.random.choice([30000, 50000, 80000, 120000, np.nan], size=n_samples)
credit_score = np.random.normal(650, 100, size=n_samples)

# สร้าง Target ที่มี Logic สัมพันธ์กับ Feature จริง
# เช่น: คนที่รายได้สูง (>60000) และ เครดิตดี (>650) มีโอกาสติด Class 1 สูงกว่า
target_prob = (np.nan_to_num(income, nan=30000) / 120000) * 0.5 + (credit_score / 800) * 0.5
target = (target_prob > 0.65).astype(int)

data = pd.DataFrame({
    'age': age,
    'income': income,
    'credit_score': credit_score,
    'education': np.random.choice(['High School', 'Bachelor', 'Master', np.nan], size=n_samples),
    'city': np.random.choice(['Bangkok', 'Chiang Mai', 'Phuket'], size=n_samples),
    'target': target
})

X = data.drop('target', axis=1)
y = data['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

NameError: name 'np' is not defined

In [ ]:
# ---------------------------------------------------------
# 2. นิยาม Feature Types และสร้าง Sub-pipelines
# ---------------------------------------------------------
num_features = ['age', 'income', 'credit_score']
cat_features = ['education', 'city']

# Pipeline สำหรับ ตัวเลข: จัดการ Missing Value -> Scale ข้อมูล
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline สำหรับ หมวดหมู่: จัดการ Missing Value -> One-Hot Encode
cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# รวม Pipeline สำหรับชนิดข้อมูลต่าง ๆ เข้าด้วยกัน
preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
])

In [ ]:
# ---------------------------------------------------------
# 3. สร้าง Full Pipeline (Preprocessing + Feature Selection)
# ---------------------------------------------------------
# สามารถนำ Preprocessor ตัวนี้ไปต่อกับ โมเดลใดก็ได้
def create_model_pipeline(classifier):
    return Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('feature_selection', SelectKBest(score_func=f_classif, k=5)), # ดึงเฉพาะ 5 features ที่ดีที่สุด
        ('classifier', classifier)
    ])

Comparison Efficeincy before and after Preprocessing

In [ ]:
# ---------------------------------------------------------
# Case A: ไม่ทำ Preprocessing / เติมค่าแบบส่งเดช (Naive Approach)
# ---------------------------------------------------------
# Logistic Regression จะพังทันทีถ้าเจอ NaN หรือ Categorical String
X_train_naive = X_train.copy()
X_test_naive = X_test.copy()

# เติมง่ายๆ แบบไม่มี Pipeline
X_train_naive['age'] = X_train_naive['age'].fillna(0)
X_train_naive['income'] = X_train_naive['income'].fillna(0)
X_train_naive['credit_score'] = X_train_naive['credit_score'].fillna(0)
X_train_naive = pd.get_dummies(X_train_naive)

X_test_naive['age'] = X_test_naive['age'].fillna(0)
X_test_naive['income'] = X_test_naive['income'].fillna(0)
X_test_naive['credit_score'] = X_test_naive['credit_score'].fillna(0)
X_test_naive = pd.get_dummies(X_test_naive)

# Align columns กันปัญหากรณีหมวดหมู่ไม่เท่ากัน
X_train_naive, X_test_naive = X_train_naive.align(X_test_naive, join='left', axis=1, fill_value=0)

raw_model = LogisticRegression()
raw_model.fit(X_train_naive, y_train)
acc_before = accuracy_score(y_test, raw_model.predict(X_test_naive))

In [ ]:
# ---------------------------------------------------------
# Case B: ใช้ Preprocessing Pipeline
# ---------------------------------------------------------
log_reg_pipeline = create_model_pipeline(LogisticRegression())
log_reg_pipeline.fit(X_train, y_train)
acc_after_lr = accuracy_score(y_test, log_reg_pipeline.predict(X_test))

# ลองใช้ Pipeline เดิมเปลี่ยนแค่โมเดลเป็น Random Forest
rf_pipeline = create_model_pipeline(RandomForestClassifier(random_state=42))
rf_pipeline.fit(X_train, y_train)
acc_after_rf = accuracy_score(y_test, rf_pipeline.predict(X_test))

In [ ]:
# ---------------------------------------------------------
# สรุปผลลัพธ์
# ---------------------------------------------------------
print(f"Accuracy (Naive Prep + Logistic Regression) : {acc_before:.4f}")
print(f"Accuracy (Pipeline + Logistic Regression)   : {acc_after_lr:.4f}")
print(f"Accuracy (Pipeline + Random Forest)         : {acc_after_rf:.4f}")